In [15]:
import pandas as pd
import numpy as np
import geopandas as gpd
import plotly.express as px
import plotly.graph_objects as go

import nbformat
import json
from urllib.request import urlopen

import shapely.geometry
from shapely.geometry import Polygon
import shapely.ops

In [10]:
with urlopen('https://raw.githubusercontent.com/plotly/datasets/master/geojson-counties-fips.json') as response:
    counties = json.load(response)
    
#drop 'StateFIPS': '09'
for i in range(0, 3214):
    if counties['features'][i]['properties']['STATE'] == '09':
        del counties['features'][i]


#add planning regions
json_file = ("CT_Planning_Regions.geojson")

with open(json_file) as f:
    ct_counties = json.load(f)

ct_counties.pop("name", None)
ct_counties.pop("crs", None)

for i in range(0, 9):
    ct_counties['features'][i]['properties']['NAME'] = ct_counties['features'][i]['properties'].pop("PlanningRegion")
    ct_counties['features'][i]['properties']['COUNTY'] = ct_counties['features'][i]['properties'].pop("PlanningRegionFIPS")
    ct_counties['features'][i]['properties']['STATE'] = ct_counties['features'][i]['properties'].pop("StateFIPS")
    ct_counties['features'][i]['properties']['GEO_ID'] = "0500000US" + ct_counties['features'][i]['properties'].pop("PlanningRegionFIPS_GEOID")
    ct_counties['features'][i]['properties'].pop("OBJECTID", None)
    ct_counties['features'][i]['properties'].pop("SHAPE__Area", None)
    ct_counties['features'][i]['properties'].pop("SHAPE__Length", None)
    ct_counties['features'][i]['properties']['LSAD'] = "County"
    ct_counties['features'][i]['properties']['CENSUSAREA'] = "None"


for i in range(0, 9):
    ct_counties['features'][i]['id'] = str(ct_counties['features'][i]['properties']['STATE'] + ct_counties['features'][i]['properties']['COUNTY'])

for i in range(0,9):
    counties['features'].append(ct_counties['features'][i])
#add oglala

oglala = { "type": "Feature", "properties": { "STATEFP": "46", "COUNTYFP": "102", "COUNTYNS": "01266992", "AFFGEOID": "0500000US46102", "GEOID": "46102", "NAME": "Oglala Lakota", "LSAD": "06", "ALAND": 5422404947, "AWATER": 7126155 }, "geometry": { "type": "Polygon", "coordinates": [ [ [ -103.001017, 43.605815999057796 ], [ -102.9025, 43.674887999057695 ], [ -102.809398, 43.687559999057683 ], [ -102.175231, 43.687755999057686 ], [ -102.139158, 43.700947999057654 ], [ -102.139051, 43.476675999058003 ], [ -102.110819, 43.389901999058168 ], [ -102.108892, 43.129257999058666 ], [ -102.082486319192, 42.99914054757825 ], [ -102.082548043072009, 42.999140639448953 ], [ -102.40864, 42.999625999058956 ], [ -102.792110939839006, 43.000036234829345 ], [ -103.000609526033003, 43.000259285833359 ], [ -103.000913, 43.476847999058016 ], [ -103.001017, 43.605815999057796 ] ] ] }, 'id': '46102'}
counties['features'].append(oglala)

In [11]:
#one with aa 

#one with state polygons

yrs = list(range(15, 26))
df_name = ["labor_" + str(x) for x in yrs]

datasets = {}
for k, v in zip(df_name, yrs):
    a = pd.read_excel(f"laucnty{str(v)}.xlsx", skiprows=1)
    datasets[k] = a

#stack data
data_frame_labor15_25 = pd.concat(datasets.values(), ignore_index=True)

#remove PR, VI + others
territories = [60, 66, 69, 72, 74, 78]
data_frame_labor15_25 = data_frame_labor15_25[~data_frame_labor15_25['State FIPS Code'].isin(territories)]


In [16]:
#add county polygon data, rename columns
counties = gpd.read_file("counties.geojson")
counties[['STATEFP', 'COUNTYFP' ]] = counties[['STATEFP', 'COUNTYFP' ]].astype(float)
counties = counties.rename(columns={'STATEFP': 'State FIPS Code', 'COUNTYFP': 'County FIPS Code'}) 

#CT has new "planning regions," replace CT counties with planning regions
json_file = ("CT_Planning_Regions.geojson")

with open(json_file) as f:
    ct_counties = json.load(f)

name = []
FIPS_code = []
polygon = []

for i in range(0, 9):
    name.append(ct_counties['features'][i]['properties']['PlanningRegion']) #county name
    FIPS_code.append(ct_counties['features'][i]['properties']['PlanningRegionFIPS']) #county FIPS code
    polygon.append(ct_counties['features'][i]['geometry']['coordinates']) #polygon

ct_counties = pd.DataFrame({"NAME":name, "County FIPS Code":FIPS_code, "State FIPS Code": 9.0, "geometry":polygon})

#function to find the polygon file within nested list
def polygon_nested_lists(a):
    count = 0
    og_a = a

    while a:
        if isinstance(a, list):
            count += 1
            a = a[0]
            continue
        else: 
            for i in range(0, count-2):
                og_a = og_a[0]
            return Polygon(og_a)

ct_counties['geometry'] = ct_counties['geometry'].apply(lambda x: polygon_nested_lists(x))
ct_counties['County FIPS Code'] = ct_counties['County FIPS Code'].astype(float)

#remove previous counties and merge
counties = counties[counties['State FIPS Code'] != 9.0] 
counties = pd.concat([counties, ct_counties], ignore_index=True)


In [20]:
counties.head()

,State FIPS Code,County FIPS Code,COUNTYNS,AFFGEOID,GEOID,NAME,LSAD,ALAND,AWATER,geometry
0,6.0,75.0,00277302,0500000US06075,06075,San Francisco,06,1.214851e+08,479107241.0,"POLYGON ((-122.51198 37.77113, -122.4654 37.80..."
1,25.0,25.0,00606939,0500000US25025,25025,Suffolk,06,1.508555e+08,160479920.0,"POLYGON ((-71.19116 42.28306, -71.15689 42.330..."
2,31.0,7.0,00835826,0500000US31007,31007,Banner,06,1.932677e+09,397069.0,"POLYGON ((-104.05283 41.69795, -103.37039 41.6..."
3,37.0,181.0,01008591,0500000US37181,37181,Vance,06,6.537058e+08,42187365.0,"POLYGON ((-78.49778 36.51448, -78.45728 36.541..."
4,48.0,421.0,01383996,0500000US48421,48421,Sherman,06,2.390651e+09,428754.0,"POLYGON ((-102.16246 36.50033, -102.03234 36.5..."


In [21]:
#data_frame_labor15_25_clean = pd.merge(data_frame_labor15_25, counties[['State FIPS Code', 'County FIPS Code', 'geometry']], on=['State FIPS Code', 'County FIPS Code'], how='inner')
data_frame_labor15_25.head()

,LAUS Code,State FIPS Code,County FIPS Code,County Name/State Abbreviation,Year,Labor Force,Employed,Unemployed,Unemployment Rate (%)
0,CN0100100000000,1.0,1.0,"Autauga County, AL",2015.0,25783.0,24441.0,1342.0,5.2
1,CN0100300000000,1.0,3.0,"Baldwin County, AL",2015.0,88323.0,83424.0,4899.0,5.5
2,CN0100500000000,1.0,5.0,"Barbour County, AL",2015.0,8670.0,7899.0,771.0,8.9
3,CN0100700000000,1.0,7.0,"Bibb County, AL",2015.0,8633.0,8061.0,572.0,6.6
4,CN0100900000000,1.0,9.0,"Blount County, AL",2015.0,24647.0,23313.0,1334.0,5.4


In [ ]:

def add_zeros(string, num):
    to_add = num - len(string)
    return str((to_add * "0") + string)

data_frame_labor15_25['id'] = data_frame_labor15_25.apply(lambda x: (add_zeros(str(x['State FIPS Code'])[:-2], 2)) + (add_zeros(str(x['County FIPS Code'])[:-2], 3)),axis=1)
data_frame_labor15_25['Year'] = data_frame_labor15_25["Year"].astype('Int64')
data_frame_labor15_25 = data_frame_labor15_25[data_frame_labor15_25['Year'].notna()]
data_frame_labor15_25['County Name'] = data_frame_labor15_25['County Name/State Abbreviation'].apply(lambda x: x[:-4])
data_frame_labor15_25['State Abbreviation'] = data_frame_labor15_25['County Name/State Abbreviation'].apply(lambda x: x[-2:])

In [ ]:
state_boundaries = gpd.read_file(r"cb_2018_us_state_500k\cb_2018_us_state_500k.shp")
state_boundaries.to_file('myJson.geojson', driver='GeoJSON')
df = data_frame_labor15_25.groupby(['State FIPS Code', 'Year'])['Unemployment Rate (%)'].mean()
df = df.to_frame(name='Unemployment Rate Avg').reset_index()
df = df.rename(columns = {'State FIPS Code': 'STATEFP'})

In [ ]:
fig = px.choropleth_map(unemployment, 
                            geojson=counties, 
                            locations='id', 
                            color='Unemployment Rate (%)',
                           color_continuous_scale="sunsetdark",
                           range_color=(0, 12),
                           map_style="carto-positron",
                           zoom=3, center = {"lat": 37.0902, "lon": -95.7129},
                           animation_frame="Year", 
                           opacity=0.7,
                           custom_data = ['County Name', "Unemployment Rate (%)"]
                          )

for frame in fig.frames:
    for trace in frame.data:
        trace.hovertemplate = '<br>'.join(["%{customdata[0]}", "%{customdata[1]}%"])

fig.update_traces(hovertemplate = '<br>'.join(["%{customdata[0]}", "%{customdata[1]}%"]))
fig.update_layout(margin={"r":0,"t":0,"l":0,"b":0})
fig.update_traces(marker_line_width=0, marker_opacity=0.8)

#make map wider

state_boundaries = gpd.read_file(r"C:\Users\annas\OneDrive\Github\plotly_bls_map\cb_2018_us_state_500k\cb_2018_us_state_500k.shp")
state_boundaries.to_file('myJson.geojson', driver='GeoJSON')

fig.update_layout(
    coloraxis_showscale=False,
    map={
        "style":"carto-positron",
        "layers": [
            {
                "source": state_boundaries["geometry"].__geo_interface__,
                "type": "line",
                "color": "black"
            }
        ]
    },
)



In [ ]:
fig.show()